# Qwen3 Local Inference

This notebook loads a Qwen3 causal language model with Hugging Face Transformers and runs single-prompt and batch chat inference.

The official Qwen3 dense model nearest to "1.6B" is `Qwen/Qwen3-1.7B`. Override `MODEL_ID` or set `QWEN3_MODEL_ID` if you need a different checkpoint, including a third-party 1.6B fine-tune.



In [ ]:
from __future__ import annotations

import gc
import os
import time
from dataclasses import dataclass
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.getenv("QWEN3_MODEL_ID", "Qwen/Qwen3-1.7B")
LOCAL_FILES_ONLY = os.getenv("HF_LOCAL_FILES_ONLY", "0") == "1"

print(f"model: {MODEL_ID}")
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")
    print(f"cuda capability: {torch.cuda.get_device_capability(0)}")


In [ ]:
@dataclass(frozen=True)
class GenerationSettings:
    max_new_tokens: int = 512
    temperature: float = 0.7
    top_p: float = 0.8
    top_k: int = 20
    do_sample: bool = True
    repetition_penalty: float = 1.05


THINKING_SETTINGS = GenerationSettings(
    max_new_tokens=1024,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    do_sample=True,
    repetition_penalty=1.05,
)

NON_THINKING_SETTINGS = GenerationSettings(
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
    repetition_penalty=1.05,
)



In [ ]:
def _from_pretrained_kwargs() -> dict[str, Any]:
    kwargs: dict[str, Any] = {
        "device_map": "auto" if torch.cuda.is_available() else None,
        "local_files_only": LOCAL_FILES_ONLY,
    }
    return {key: value for key, value in kwargs.items() if value is not None}


def load_qwen3(model_id: str = MODEL_ID):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        local_files_only=LOCAL_FILES_ONLY,
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = _from_pretrained_kwargs()
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype="auto",
            trust_remote_code=True,
            **kwargs,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype="auto",
            trust_remote_code=True,
            **kwargs,
        )

    model.eval()
    return tokenizer, model


t0 = time.perf_counter()
tokenizer, model = load_qwen3()
print(f"loaded in {time.perf_counter() - t0:.1f}s")
print(f"model device: {next(model.parameters()).device}")



In [ ]:
def _first_model_device() -> torch.device:
    return next(model.parameters()).device


def format_chat(
    messages: list[dict[str, str]],
    *,
    enable_thinking: bool = False,
    tokenize: bool = True,
):
    kwargs: dict[str, Any] = {
        "add_generation_prompt": True,
        "tokenize": tokenize,
    }
    if tokenize:
        kwargs["return_tensors"] = "pt"
    try:
        return tokenizer.apply_chat_template(
            messages,
            enable_thinking=enable_thinking,
            **kwargs,
        )
    except TypeError:
        # Older chat templates may not expose enable_thinking.
        return tokenizer.apply_chat_template(messages, **kwargs)


def split_qwen3_response(text: str) -> dict[str, str]:
    if "</think>" not in text:
        return {"thinking": "", "answer": text.strip()}

    before, answer = text.split("</think>", 1)
    thinking = before.replace("<think>", "", 1).strip()
    return {"thinking": thinking, "answer": answer.strip()}


@torch.inference_mode()
def generate_chat(
    messages: list[dict[str, str]],
    *,
    settings: GenerationSettings = NON_THINKING_SETTINGS,
    enable_thinking: bool = False,
) -> dict[str, Any]:
    input_ids = format_chat(messages, enable_thinking=enable_thinking, tokenize=True).to(
        _first_model_device()
    )

    output_ids = model.generate(
        input_ids,
        max_new_tokens=settings.max_new_tokens,
        temperature=settings.temperature,
        top_p=settings.top_p,
        top_k=settings.top_k,
        do_sample=settings.do_sample,
        repetition_penalty=settings.repetition_penalty,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = output_ids[0, input_ids.shape[-1] :]
    raw_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    parsed = split_qwen3_response(raw_text)
    parsed["raw"] = raw_text
    return parsed


@torch.inference_mode()
def batch_generate_chat(
    prompts: list[str],
    *,
    system_prompt: str | None = None,
    settings: GenerationSettings = NON_THINKING_SETTINGS,
    enable_thinking: bool = False,
) -> list[dict[str, Any]]:
    messages_batch = []
    for prompt in prompts:
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        messages_batch.append(messages)

    prompt_texts = [
        format_chat(messages, enable_thinking=enable_thinking, tokenize=False)
        for messages in messages_batch
    ]
    inputs = tokenizer(prompt_texts, return_tensors="pt", padding=True).to(_first_model_device())

    output_ids = model.generate(
        **inputs,
        max_new_tokens=settings.max_new_tokens,
        temperature=settings.temperature,
        top_p=settings.top_p,
        top_k=settings.top_k,
        do_sample=settings.do_sample,
        repetition_penalty=settings.repetition_penalty,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated = output_ids[:, inputs["input_ids"].shape[-1] :]
    raw_texts = tokenizer.batch_decode(generated, skip_special_tokens=True)
    results = []
    for prompt, raw_text in zip(prompts, raw_texts, strict=True):
        parsed = split_qwen3_response(raw_text)
        parsed["prompt"] = prompt
        parsed["raw"] = raw_text
        results.append(parsed)
    return results



In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a concise assistant. Answer directly.",
    },
    {
        "role": "user",
        "content": "Explain the difference between CTR and conversion rate in one paragraph.",
    },
]

result = generate_chat(messages, settings=NON_THINKING_SETTINGS, enable_thinking=False)
print(result["answer"])



In [ ]:
thinking_messages = [
    {
        "role": "user",
        "content": (
            "A campaign has 20,000 impressions, 620 clicks, and 31 purchases. "
            "Compute CTR and purchase conversion rate."
        ),
    },
]

thinking_result = generate_chat(
    thinking_messages,
    settings=THINKING_SETTINGS,
    enable_thinking=True,
)

print("Answer:")
print(thinking_result["answer"])

if thinking_result["thinking"]:
    print("\nThinking trace:")
    print(thinking_result["thinking"])


In [ ]:
prompts = [
    "Write a short feature description for an ad click prediction model.",
    "List three common causes of poor CTR prediction calibration.",
    "Give one practical way to monitor model drift for ad auctions.",
]

batch_results = batch_generate_chat(
    prompts,
    system_prompt="You are a practical machine learning assistant.",
    settings=GenerationSettings(max_new_tokens=192),
    enable_thinking=False,
)

for item in batch_results:
    print(f"Prompt: {item['prompt']}")
    print(item["answer"])
    print("-" * 80)



In [ ]:
def free_model_memory() -> None:
    global model, tokenizer
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Call free_model_memory() when you are done with inference in this notebook.

